# 06b — MoE Geo Router: Train Sub-Models (HCM Expert + Generalist)

**Ý tưởng:** Thị trường HCM (chung cư/phòng trọ, thanh khoản nhanh) và các tỉnh khác
(đất nền, thanh khoản chậm) có pattern khác nhau. Ép 1 model học cả 2 sẽ bị "trung bình hóa".

**Thiết kế:**
- **Router**: dựa vào `u_top_city` từ lịch sử clickstream → phân luồng user
- **Sub-Model A (HCM Expert)**: train trên user × item trong HCM
- **Sub-Model B (Generalist)**: train trên user × item ngoài HCM
  (+ 20% undersample từ HCM để giữ cross-market generalisation)

Output: `cache_drive/model_hcm.txt`, `cache_drive/model_general.txt`,
`cache_drive/model_hcm_feats.json`, `cache_drive/model_general_feats.json`


In [ ]:
print('Skipping pip install (local kernel).')

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from hashlib import md5

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
os.makedirs(CACHE_DIR, exist_ok=True)
for p in (TRAINING_DIR, PROJECT_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from utils.features import build_user_features, build_item_features, add_cross_features
from utils.metrics import mean_recall_at_k, mean_ndcg_at_k
print("Imports OK | CACHE_DIR:", CACHE_DIR)


In [ ]:
TRAIN_DATE_END = "2026-04-09"
VALID_START    = "2026-03-13"

HCM_CITY_NAMES = {
    "Hồ Chí Minh", "Ho Chi Minh", "TP. Hồ Chí Minh", "TP.HCM",
    "TP Hồ Chí Minh", "Thành phố Hồ Chí Minh",
}

MONO_MAP = {
    "age_at_train_end":         -1,
    "recency_evt_days":         -1,
    "u_recency_days":           -1,
    "i_CR_30d":                  1,
    "i_contacts_24h_mean_30d":   1,
    "i_n_pos_30d":               1,
    "i_n_pos_7d":                1,
    "u_n_pos_30d":               1,
    # Super features
    "i_velocity_contacts":       1,
    "i_velocity_views":          1,
    "x_price_affinity":          1,
}
DROP_COLS = {"user_id","item_id","label","title","posted_date","expected_expired_date",
             "first_evt_date","last_evt_date","last_snap_date","project_id","_h","_geo"}

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}
print("Constants OK | HCM aliases:", len(HCM_CITY_NAMES))

In [ ]:
# ---- Load train matrix (output of nb05) ----
t0 = time.time()
full = pd.read_parquet(f"{CACHE_DIR}/train_matrix_valid.parquet")
print(f"full matrix: {full.shape} | {time.time()-t0:.1f}s")

# Load dim to get city_name per item
dim = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet",
                       columns=["item_id","city_name"])
full = full.merge(dim, on="item_id", how="left")
print(f"city_name null: {full['city_name'].isna().sum():,}")


In [ ]:
# ---- Router: classify each row's city segment ----
# Use city_name from item (listing location), not user's top_city,
# so the split is about ITEM geography (where the listing is).
full["_geo"] = full["city_name"].apply(
    lambda c: "HCM" if (isinstance(c, str) and c.strip() in HCM_CITY_NAMES) else "OTHER"
)
geo_counts = full["_geo"].value_counts()
print("Geo distribution in train matrix:")
print(geo_counts)
print(f"HCM share: {geo_counts.get('HCM',0)/len(full)*100:.1f}%")


In [ ]:
def build_and_train(df, seg_label, save_model_path, save_feats_path,
                    n_boost=2000, early_stop=100):
    """Train a lambdarank LightGBM on dataframe df, return model."""
    cat_cols  = [c for c in df.columns if c not in DROP_COLS and df[c].dtype == "object"]
    num_cols  = [c for c in df.columns if c not in DROP_COLS and df[c].dtype != "object"
                 and "datetime" not in str(df[c].dtype)]
    feat_cols = cat_cols + num_cols
    for c in cat_cols:
        df[c] = df[c].astype("category")

    df["_h"] = df["user_id"].map(lambda s: int(md5(str(s).encode()).hexdigest(), 16) % 100)
    tr_m = df["_h"] < 80;  va_m = df["_h"] >= 80
    X_tr = df.loc[tr_m, feat_cols];  y_tr = df.loc[tr_m, "label"].values
    g_tr = df.loc[tr_m].groupby("user_id", sort=False).size().values
    X_va = df.loc[va_m, feat_cols];  y_va = df.loc[va_m, "label"].values
    g_va = df.loc[va_m].groupby("user_id", sort=False).size().values

    mono_list = [MONO_MAP.get(c, 0) for c in feat_cols]
    active_mono = sum(v != 0 for v in mono_list)
    print(f"  [{seg_label}] rows={len(df):,} | feats={len(feat_cols)} "
          f"(cat={len(cat_cols)}) | mono_active={active_mono}")
    print(f"  train groups={len(g_tr)} pos={y_tr.sum()} | "
          f"val groups={len(g_va)} pos={y_va.sum()}")

    params = dict(
        objective="lambdarank", metric="ndcg", ndcg_eval_at=[10],
        learning_rate=0.05, num_leaves=127, min_data_in_leaf=100,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        lambda_l2=1.0, max_bin=255, seed=42, verbosity=-1,
        monotone_constraints=mono_list,
        monotone_constraints_method="advanced",
    )
    d_tr = lgb.Dataset(X_tr, label=y_tr, group=g_tr,
                       categorical_feature=cat_cols, free_raw_data=False)
    d_va = lgb.Dataset(X_va, label=y_va, group=g_va,
                       categorical_feature=cat_cols, reference=d_tr, free_raw_data=False)

    t0 = time.time()
    model = lgb.train(
        params, d_tr, num_boost_round=n_boost,
        valid_sets=[d_va], valid_names=["valid"],
        callbacks=[lgb.early_stopping(early_stop), lgb.log_evaluation(100)],
    )
    print(f"  [{seg_label}] trained {time.time()-t0:.0f}s | best_iter={model.best_iteration}")
    model.save_model(save_model_path)
    with open(save_feats_path, "w") as fj:
        json.dump(feat_cols, fj)

    # Internal eval on va split
    preds = {}
    df.loc[va_m, "_pred"] = model.predict(X_va, num_iteration=model.best_iteration)
    for uid, grp in df[va_m].groupby("user_id", sort=False):
        preds[uid] = grp.nlargest(10, "_pred")["item_id"].tolist()
    valid_gt = pd.read_parquet(f"{CACHE_DIR}/valid_gt.parquet")
    gt_dict = valid_gt.groupby("user_id")["item_id"].agg(set).to_dict()
    gt_va = {u: gt_dict[u] for u in preds if u in gt_dict}
    r10 = mean_recall_at_k(preds, gt_va, k=10)
    n10 = mean_ndcg_at_k(preds,  gt_va, k=10)
    print(f"  [{seg_label}] INTERNAL (val split) => Recall@10={r10:.4f}  NDCG@10={n10:.4f}")
    return model, feat_cols


In [ ]:
# ---- Sub-Model A: HCM Expert ----
df_hcm = full[full["_geo"] == "HCM"].copy()
print(f"HCM subset: {len(df_hcm):,} rows")

model_hcm, feats_hcm = build_and_train(
    df_hcm, "HCM",
    save_model_path=f"{CACHE_DIR}/model_hcm.txt",
    save_feats_path=f"{CACHE_DIR}/model_hcm_feats.json",
)


In [ ]:
# ---- Sub-Model B: Generalist (non-HCM + 20% HCM undersample) ----
df_other = full[full["_geo"] == "OTHER"].copy()
n_hcm_sample = max(1, int(len(df_other) * 0.20))
df_hcm_sample = df_hcm.sample(n=min(n_hcm_sample, len(df_hcm)), random_state=42)
df_general = pd.concat([df_other, df_hcm_sample], ignore_index=True)
print(f"Generalist subset: {len(df_general):,} rows "
      f"(other={len(df_other):,} + hcm_sample={len(df_hcm_sample):,})")

model_general, feats_general = build_and_train(
    df_general, "GENERAL",
    save_model_path=f"{CACHE_DIR}/model_general.txt",
    save_feats_path=f"{CACHE_DIR}/model_general_feats.json",
)


In [ ]:
print("\nMoE models saved:")
for fname in ["model_hcm.txt","model_general.txt",
              "model_hcm_feats.json","model_general_feats.json"]:
    p = f"{CACHE_DIR}/{fname}"
    size_mb = os.path.getsize(p) / 1024**2
    print(f"  {fname}: {size_mb:.1f} MB")
